# Getting Started with `deph`

`deph` is a powerful tool for analyzing Python code dependencies and isolating self-contained, minimal code snippets. This notebook will walk you through the two core functionalities:

1.  **`deph.analyze()`**: To inspect the dependencies of your functions and classes.
2.  **`deph.isolate()`**: To generate a portable, standalone Python script from your code.

## 1. Setup

First, let's define some example functions and classes that we'll use for our analysis. These examples include various types of dependencies:

- Standard library imports (`os`, `json`)
- Third-party library imports (`numpy`)
- Local function calls
- Module-level variables

In [ ]:
import os
import numpy as np

# A module-level variable
DEFAULT_PREFIX = "processed_"

def get_file_path(directory: str, filename: str) -> str:
    """Constructs a full file path."""
    return os.path.join(directory, filename)

def process_data(data: np.ndarray, prefix: str = None) -> dict:
    """Processes a numpy array and returns statistics."""
    import json # Local import
    
    if prefix is None:
        prefix = DEFAULT_PREFIX
    
    # Call another local function
    output_path = get_file_path("/tmp", f"{prefix}output.json")
    
    stats = {
        "mean": np.mean(data),
        "std_dev": np.std(data),
        "shape": data.shape,
        "output_file": output_path
    }
    
    return json.dumps(stats, indent=2)

Now, let's import the `deph` library. If you are running this from the project's root directory, you may need to add the `src` folder to your Python path.

In [ ]:
import sys
from pathlib import Path

# Add the 'src' directory to the path to find the 'deph' package
src_path = str(Path.cwd() / 'src')
if src_path not in sys.path:
    sys.path.insert(0, src_path)

import deph

## 2. Analyzing Dependencies with `deph.analyze()`

The `analyze` function inspects a target object (like our `process_data` function) and returns a detailed report of its dependencies. This is useful for understanding what a piece of code needs to run.

In [ ]:
# Analyze the process_data function
report = deph.analyze(process_data)

# The report is a special dictionary that pretty-prints its contents
report

### Understanding the Analysis Report

The report contains several key sections:

- **`entries`**: The initial object(s) you asked to analyze.
- **`def_items`**: A list of all function and class definitions that the entry depends on. Notice how it automatically included `get_file_path` because `process_data` calls it.
- **`imports`**: A breakdown of all required imports, categorized by module. It correctly identifies `numpy` and `os`.
- **`vars`**: Any module-level variables that are used, like `DEFAULT_PREFIX`.
- **`unbound`**: A list of names that were used but could not be resolved. This should be empty if all dependencies are found.

## 3. Isolating Code with `deph.isolate()`

The `isolate` function takes the analysis one step further: it generates a complete, self-contained Python script that includes everything needed to run your target function. This is perfect for sharing, testing, or deploying a piece of logic.

In [ ]:
code, warnings, report = deph.isolate(process_data)

Let's print the generated code to see what it looks like.

In [ ]:
print(code)

### Understanding the Isolated Code

The generated script is organized logically:

1.  **Header**: A comment block that lists the entry points and any required packages (e.g., `pip install numpy`).
2.  **Imports**: All necessary `import` statements are placed at the top.
3.  **Variables**: Required module-level variables like `DEFAULT_PREFIX` are included.
4.  **Definitions**: All dependent functions and classes (`get_file_path` and `process_data`) are included in the correct order.

This script can now be saved to a `.py` file and run anywhere, as long as the required packages are installed.

## 4. Next Steps

You've now seen the basic workflow of `deph`. You can try it on your own functions and classes, including those with more complex dependencies. `deph` is a powerful utility for understanding, refactoring, and sharing your Python code.